# 08-1 空間流病：樓層翼區侵襲率與 Spot Map

長官問：「在哪裡最嚴重？」

松柏護理之家有 3 層樓 × 2 翼區（A / B），共 280 位住民。
我們要找出哪些區域侵襲率最高，並畫出 spot map。

流程：**資料準備 → floor × wing 侵襲率 → 熱力圖 → 每間房侵襲率 → Spot Map → 致死率空間比較**

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
# --- Step 1: 資料準備 ---
import pathlib

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# -- CJK font setup (避免中文標籤顯示為方框) --
# 掃描系統字型目錄，顯式註冊 CJK 字型（比依賴快取更可靠）
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)
df["died"] = (df["outcome"] == "died").astype(int)

print(f"住民數：{len(df)}")
print(f"感染：{df['infected'].sum()}")
print(f"死亡：{df['died'].sum()}")
print(f"\n樓層：{sorted(df['floor'].unique())}")
print(f"翼區：{sorted(df['wing'].unique())}")
print(f"房間數：{df['room'].nunique()}")

In [ ]:
# --- Step 2: Floor × Wing 侵襲率 ---
spatial = df.groupby(["floor", "wing"]).agg(
    total=("case_id", "count"),
    infected=("infected", "sum"),
    died=("died", "sum"),
).reset_index()
spatial["attack_rate"] = (spatial["infected"] / spatial["total"] * 100).round(1)
spatial["cfr"] = (spatial["died"] / spatial["infected"] * 100).round(1)

print("=== 樓層翼區統計 ===")
print(spatial.to_string(index=False))

In [ ]:
# --- Step 3: 侵襲率熱力圖 ---
heatmap_ar = spatial.pivot(index="floor", columns="wing", values="attack_rate")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# 侵襲率
sns.heatmap(heatmap_ar, annot=True, fmt=".1f", cmap="YlOrRd",
            cbar_kws={"label": "%"}, ax=axes[0])
axes[0].set_title("侵襲率 (%) by Floor \u00d7 Wing")
axes[0].set_ylabel("Floor")

# 致死率
heatmap_cfr = spatial.pivot(index="floor", columns="wing", values="cfr")
sns.heatmap(heatmap_cfr, annot=True, fmt=".1f", cmap="Reds",
            cbar_kws={"label": "%"}, ax=axes[1])
axes[1].set_title("致死率 (%) by Floor \u00d7 Wing")
axes[1].set_ylabel("Floor")

plt.tight_layout()
plt.show()

print("\u2192 侵襲率最高的區域：")
top = spatial.nlargest(3, "attack_rate")
for _, row in top.iterrows():
    print(f"  {row['floor']}F-{row['wing']} 翼：{row['attack_rate']}%")

In [ ]:
# --- Step 4: 每間房侵襲率 ---
room_stats = df.groupby("room").agg(
    total=("case_id", "count"),
    infected=("infected", "sum"),
).reset_index()
room_stats["attack_rate"] = (room_stats["infected"] / room_stats["total"] * 100).round(1)

# 解析 room 名稱（例如 "2A-03" → floor=2, wing=A, room_num=3）
room_stats["floor_num"] = room_stats["room"].str[0].astype(int)
room_stats["wing_code"] = room_stats["room"].str[1]
room_stats["room_num"] = room_stats["room"].str.split("-").str[1].astype(int)

print(f"共 {len(room_stats)} 間房")
print(f"\n侵襲率 100% 的房間（全部住民都感染）：")
full = room_stats[room_stats["attack_rate"] == 100.0]
print(f"  共 {len(full)} 間")
print(f"\n侵襲率 0% 的房間（無人感染）：")
zero = room_stats[room_stats["attack_rate"] == 0.0]
print(f"  共 {len(zero)} 間")

In [ ]:
# --- Step 5: Spot Map（模擬護理之家平面圖）---
# X 軸 = 房間號碼，A 翼在左半、B 翼在右半
# Y 軸 = 樓層
max_room_a = room_stats[room_stats["wing_code"] == "A"]["room_num"].max()
gap = 5  # A 翼和 B 翼之間的間距

room_stats["x"] = room_stats.apply(
    lambda r: r["room_num"] if r["wing_code"] == "A"
    else r["room_num"] + max_room_a + gap,
    axis=1,
)

fig, ax = plt.subplots(figsize=(14, 5))
sc = ax.scatter(
    room_stats["x"],
    room_stats["floor_num"],
    s=room_stats["total"] * 50,
    c=room_stats["attack_rate"],
    cmap="YlOrRd",
    edgecolors="black",
    linewidth=0.5,
    alpha=0.8,
    vmin=0,
    vmax=100,
)
plt.colorbar(sc, label="侵襲率 (%)")

# 分隔線與標籤
mid_x = max_room_a + gap / 2
ax.axvline(x=mid_x, color="gray", linestyle="--", alpha=0.5)
ax.text(max_room_a / 2, 3.5, "A 翼", ha="center", fontsize=12, fontweight="bold")
ax.text(max_room_a + gap + 12, 3.5, "B 翼", ha="center", fontsize=12, fontweight="bold")

ax.set_yticks([1, 2, 3])
ax.set_yticklabels(["1F", "2F", "3F"])
ax.set_xlabel("房間號碼")
ax.set_ylabel("樓層")
ax.set_title("Spot Map \u2014 每間房的侵襲率（圓點大小 = 住民數，顏色 = 侵襲率）")
plt.tight_layout()
plt.show()

print("\u2192 深色 (高侵襲率) 圓點是否集中在某些翼區？")
print("\u2192 這些高風險區域可能共用汙染的熱水管線或蓮蓬頭")

In [ ]:
# --- Step 6: 翼區侵襲率排序條圖 ---
spatial["label"] = spatial["floor"].astype(str) + "F-" + spatial["wing"]
spatial_sorted = spatial.sort_values("attack_rate", ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(
    spatial_sorted["label"],
    spatial_sorted["attack_rate"],
    color=["#e34a33" if ar > 50 else "#2c7fb8" for ar in spatial_sorted["attack_rate"]],
)
for bar, val in zip(bars, spatial_sorted["attack_rate"]):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
            f"{val}%", va="center")

ax.set_xlabel("侵襲率 (%)")
ax.set_title("各翼區侵襲率（紅色 > 50%）")
ax.set_xlim(0, 70)
plt.tight_layout()
plt.show()

print("\u2192 侵襲率超過 50% 的翼區需要優先進行環境採檢")

## 小結

| 步驟 | 學到的技能 |
|------|------------|
| floor × wing 侵襲率 | `groupby().agg()` 多指標計算 |
| 熱力圖 | `sns.heatmap()` + `pivot()` |
| 每間房分析 | 字串解析 `room` 欄位 |
| Spot Map | `scatter()` 大小=住民、顏色=侵襲率 |
| 排序條圖 | 用顏色標記高風險區域 |

**結論**：2F 和 3F-B 翼侵襲率最高，這些區域可能共用被汙染的熱水系統。
下一個 notebook（`08_spatial_choropleth`）示範地理 choropleth 的概念。